# 🧼 Universal VCT Dataset Cleaner (By Year)
This notebook loads, cleans, and exports the Valorant Champions Tour dataset by year. It's designed to be reusable for any project using the standard Kaggle dataset.

## 📦 Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import os

## 📁 Step 2: Define File Paths and Load Data

In [ ]:
DATA_DIR = 'vct_data'

# Define paths
matches_fp = os.path.join(DATA_DIR, 'vct_matches.csv')
maps_fp = os.path.join(DATA_DIR, 'vct_maps.csv')
players_fp = os.path.join(DATA_DIR, 'vct_players.csv')

# Verify existence
for fp in [matches_fp, maps_fp, players_fp]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file: {fp}")

# Load CSVs
matches = pd.read_csv(matches_fp)
maps = pd.read_csv(maps_fp)
players = pd.read_csv(players_fp)

## 🧹 Step 3: Clean and Merge Data

In [ ]:
# Convert dates
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
players['date'] = pd.to_datetime(players['date'], errors='coerce')

# Add year columns
matches['year'] = matches['date'].dt.year
players['year'] = players['date'].dt.year

# Drop rows with nulls
matches.dropna(subset=['match_id', 'team_1', 'team_2'], inplace=True)
maps.dropna(subset=['match_id', 'map'], inplace=True)
players.dropna(subset=['match_id', 'agent'], inplace=True)

# Merge player → map → match
player_maps = pd.merge(players, maps, on='match_id', how='left')
curated = pd.merge(player_maps, matches, on='match_id', how='left')
curated['year'] = curated['date'].dt.year

## 🧠 Step 4: Map Agents to Roles

In [ ]:
role_map = {
    'Jett': 'Duelist', 'Reyna': 'Duelist', 'Raze': 'Duelist',
    'Omen': 'Controller', 'Brimstone': 'Controller', 'Viper': 'Controller',
    'Killjoy': 'Sentinel', 'Cypher': 'Sentinel', 'Chamber': 'Sentinel',
    'Sova': 'Initiator', 'Breach': 'Initiator', 'Skye': 'Initiator', 'Fade': 'Initiator'
}
curated['role'] = curated['agent'].map(role_map)

## 💾 Step 5: Export Cleaned Data By Year

In [ ]:
output_dir = 'curated_output'
os.makedirs(output_dir, exist_ok=True)

# Save each year separately
for year in sorted(curated['year'].dropna().unique()):
    year_df = curated[curated['year'] == year]
    year_path = os.path.join(output_dir, f'curated_vct_{int(year)}.csv')
    year_df.to_csv(year_path, index=False)
    print(f"✅ Saved: {year_path} ({len(year_df)} rows)")

## 📦 Step 6: Save Full Master Curated Dataset

In [ ]:
curated.to_csv(os.path.join(output_dir, 'curated_vct_all_years.csv'), index=False)
print("🎉 Saved full master file: curated_vct_all_years.csv")